# `expansion_hunter_05_submit` — queue the next wave

Run **in a VWB JupyterLab app** after
[`expansion_hunter_03_budget.ipynb`](expansion_hunter_03_budget.ipynb)
has written `keep.csv`. Run All. There is no `DISPATCH_*` switch.

Each run polls the ledger, then submits only what is ready:

- Minicram for samples that have never been submitted, or whose task never
  started (a sibling failure can leave tasks unstarted inside one Batch job).
- Genotype for samples whose minicram objects exist and whose genotype has
  not been submitted.
- Retries of samples that actually failed, and only once no first attempt
  remains in that stage.

Samples that already have their objects are never submitted again. Samples
sitting in a queued or running job are skipped. Re-run this notebook to queue
genotype after minicram, and to pick up stragglers.

[`expansion_hunter_04_dispatch.ipynb`](expansion_hunter_04_dispatch.ipynb)
is the manual version (it retries every missing sample as soon as you turn
`DISPATCH_*` on). Screenshot progress from
[`expansion_hunter_06_progress.ipynb`](expansion_hunter_06_progress.ipynb).


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "dispatch.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "dispatch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, dispatch, gcs, poll, registry, wave  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")
RUN_ID = os.environ.get("EH_RUN_ID", "eh-260929-pilot").strip()
if not RUN_ID:
    raise SystemExit("set RUN_ID / EH_RUN_ID")

OUT_PREFIX = os.environ.get(
    "EH_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
)
PRINT_READS_DOCKER = os.environ.get(
    "EH_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
EH_DOCKER = os.environ.get(
    "EH_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-expansion-hunter:0.1.0",
)
SHARD_SIZE = int(os.environ.get("EH_SHARD_SIZE", "200"))
PARALLELISM = int(os.environ.get("EH_DISPATCH_PARALLELISM", "200"))
MAX_RUN_DURATION = os.environ.get("EH_MAX_RUN_DURATION", "28800s")
USE_TUNED_COMPUTE = True

PATHS = registry.run_paths(
    output_bucket=OUTPUT_BUCKET_GS.rstrip("/"),
    pipeline="expansion_hunter",
    run_id=RUN_ID,
)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_ID:", RUN_ID)
print("ledger:", PATHS.prefix)
print("keep.csv:", PATHS.keep_csv)
print("OUT_PREFIX:", OUT_PREFIX)
print("SHARD_SIZE:", SHARD_SIZE, "PARALLELISM:", PARALLELISM)
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("gcloud:", shutil.which("gcloud"))


## Submit the next wave

Polls every ledger job, writes `status.tsv`, then submits. Safe to re-run.
A failed sample waits until every first attempt in that stage has been submitted.


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
project, sa = alloc.require_vwb_env()
if not gcs.exists(PATHS.keep_csv):
    raise SystemExit(f"missing keep.csv: {PATHS.keep_csv} (run expansion_hunter_03_budget)")

local_keep = SCRATCH / f"{RUN_ID}.keep.csv"
gcs.download_file(PATHS.keep_csv, local_keep)
keep_records = cohort.read_csv(local_keep)
by_id = {r["sample_id"]: r for r in keep_records}
sample_ids = [r["sample_id"] for r in keep_records]
print("keep:", len(keep_records))

run_meta = registry.load_run_json(PATHS)
COMPUTE = run_meta.get("compute") or {}
events = registry.load_events(PATHS)
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
print("listed objects:", len(listing))

polls: dict = {}
status_rows: list = []
seen: set = set()
now = registry.utcnow()
for ev in events:
    if ev.get("event") != "job_submitted":
        continue
    job_id = ev.get("job_id")
    if not job_id or job_id in seen:
        continue
    seen.add(job_id)
    stage = ev["stage"]
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("poll failed", job_id, str(exc).splitlines()[0][:160])
        polls[job_id] = {"state": "UNKNOWN", "tasks": []}
        continue
    polls[job_id] = {"state": polled["state"], "tasks": polled["tasks"]}
    registry.append_event(PATHS, "job_polled", stage=stage, job_id=job_id, state=polled["state"])
    status_rows.extend(
        poll.reconcile_shard(
            sample_ids=ev.get("sample_ids") or [],
            stage=stage,
            job_id=job_id,
            shard=int(ev.get("shard") or 0),
            poll=polled,
            expected=lambda sid, st, stage=stage: eh.expected_objects(sid, st, OUT_PREFIX),
            now=now,
            listing=listing,
        )
    )
if status_rows:
    registry.write_status(PATHS, status_rows)
    print("status:", PATHS.status_tsv)


def objects_done(stage: str) -> set[str]:
    return {
        sid
        for sid in sample_ids
        if poll.objects_ok(eh.expected_objects(sid, stage, OUT_PREFIX), listing)
    }


def categories_for(stage: str, done_ids: set[str]) -> dict[str, str]:
    return wave.classify_stage(
        sample_ids,
        submissions=wave.latest_submissions(events, stage),
        polls=polls,
        done_ids=done_ids,
    )


mini_done = objects_done("minicram")
gt_done = objects_done("genotype")
mini_cat = categories_for("minicram", mini_done)
gt_cat = categories_for("genotype", gt_done)
mini_todo = wave.choose_to_submit(sample_ids, mini_cat)
gt_eligible = [sid for sid in sample_ids if sid in mini_done]
gt_todo = wave.choose_to_submit(gt_eligible, gt_cat)


def report(stage: str, cats: dict[str, str], todo: list[str], eligible: list[str]) -> None:
    counts = wave.count_categories(cats)
    kind = "first attempts" if any(cats.get(sid) == "not_started" for sid in eligible) else "retries"
    held = sum(1 for sid in eligible if cats.get(sid) == "failed")
    print(
        f"{stage}: done {counts['done']}  running {counts['running']}  "
        f"failed {counts['failed']}  not started {counts['not_started']}"
    )
    if todo and kind == "first attempts" and held:
        print(f"{stage}: holding {held} failed samples until first attempts are submitted")
    print(f"{stage}: submitting {len(todo)} {kind if todo else 'samples'}")


report("minicram", mini_cat, mini_todo, sample_ids)
report("genotype", gt_cat, gt_todo, gt_eligible)


def submit_ids(stage: str, todo: list[str]) -> list[str]:
    if not todo:
        return []
    records = [by_id[sid] for sid in todo]
    image = PRINT_READS_DOCKER if stage == "minicram" else EH_DOCKER
    compute = COMPUTE.get(stage) if USE_TUNED_COMPUTE else None
    n_existing = sum(
        1
        for ev in events
        if ev.get("event") == "job_submitted" and ev.get("stage") == stage
    )
    job_ids: list[str] = []
    for shard, chunk in cohort.shards(records, SHARD_SIZE):
        rows = eh.env_rows(chunk, stage=stage, out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS,
            stage=stage,
            shard=n_existing + shard,
            records=chunk,
            env_rows=rows,
            build_job=eh.build_job,
            image=image,
            out_prefix=OUT_PREFIX,
            project=project,
            region=REGION,
            sa=sa,
            parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION,
            scratch=SCRATCH,
            compute=compute,
        )
        job_id = dispatch.maybe_submit(
            prepared=prepared,
            paths=PATHS,
            stage=stage,
            shard=n_existing + shard,
            project=project,
            region=REGION,
            do_submit=True,
        )
        if job_id:
            job_ids.append(job_id)
    return job_ids


mini_jobs = submit_ids("minicram", mini_todo)
gt_jobs = submit_ids("genotype", gt_todo)
print("submitted minicram:", mini_jobs or "(none)")
print("submitted genotype:", gt_jobs or "(none)")
